## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [3]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [4]:
sao_URL = "https://www.polyu.edu.hk/sao/"
start_idx, stop_idx = 36700, -900

docs = []

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    return text

sao_loader = RecursiveUrlLoader(
    max_depth=6,
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"News-and-Events", 
        sao_URL+"news-and-events",
        sao_URL+"Sitemap",
        sao_URL+"sitemap",
        sao_URL+"Search-Result",
        sao_URL+"search-result",
        sao_URL+"Personal-Information-Collection-Statement",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    
    doc.page_content = doc.page_content[start_idx:stop_idx]
    docs.append(doc)

https://www.polyu.edu.hk/sao/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/wellness-centre/fitness-consultation-and-services/fitness-consultation/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/non-local-student-services/programmes-for-nls/host-family-scheme-/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/non-local-student-services/living-and-working-in-hong-kong/smoking-public-health-ordinance/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/non-local-student-services/settling-in/registration-and-student-id-card/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/scholarships/application/admin-by-outside-organisations/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/wellness-centre/eim-oc-gold-campus-award/
https://www.polyu.edu.hk/sao/student-development-section/polyu-student-organisations/3election-of-executive-committee/
https://www.polyu.edu.hk/sao/counselling-and-wellness-se

In [5]:
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(docs[65].page_content[:])
print(docs[65].metadata.get('source'))

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 266
 


                                External Activities
                            


                                New Student Orientation
                            


                                About SAO
                            
Open / close

back

About SAO


                                Welcome Message
                            


                                Vision and Mission
                            


                                6 Dimensions of Wellness
                            


                                Management Team
                            


                                Equality and Diversity Awareness
                            
Open / close menu


                                Mandatory Online Training Module on Preventing Sexual Harassment on Campus
                            


                                Contact Us
                            

Quick Access

Start main cont

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU SAO Website URL: {source} ---\n\n{chunk.page_content}"

In [7]:
print(chunks[10])

page_content='--- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/wellness-centre/fitness-consultation-and-services/fitness-consultation/ ---

Contact Us
                            

Quick Access

Start main content


													Home
												


													Counselling and Wellness
												


													Wellness Centre
												


													Fitness Consultation and Services
												


													Fitness Consultation
												

Fitness Consultation

The Counselling and Wellness Section (CWS) of the Student Affairs Office (SAO) strives to enhance the psychological, physical and spiritual wellness of PolyU students. The establishment of the Wellness Centre is one of the very important initiatives to promote physical wellness of the PolyU Community. Our qualified exercise professionals will assess the physical activity of every visit of the PolyU community and provide exercise prescription to improve the health and fitness am

### 3. Document Embedding in Chroma

In [ ]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "sao_documents" if not SINGLE else "vaa_documents"

In [13]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

0

In [ ]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 1484 chunks into ChromaDB to sao_documents


### 4. Simple Testing

In [11]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_82830/954563891.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Content: --- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/student-resources-and-support-section/residential-life/hall-admission/hall-applications/ ---

Contact Us
                            

Quick Access

Start main content


													Home
												


													Student Resources and Support
												


													Residential Life
												


													Hall Admission
												


													Hall Applications
												

Hall Applications

Students are strongly encouraged to have hall-life experience during their university years, which is both memorable and rewarding. The Student Halls of Residence do not just provide students with an accommodation, but a vibrant community with abundant opportunities for them to grow and learn.

 
Application for Hall Residence 2025/26 – Full-time undergraduate students

Student Type

Application Period

Current students
(including Special Readmission Scheme (SRS) &
Readmission Scheme of CURI Residential College (RSCR